In [1]:
import subprocess
import sys
import os
import warnings
warnings.filterwarnings('ignore')

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])


required_packages = {
    'timm': 'timm',
    'torch': 'torch',
    'torchvision': 'torchvision',
    'opencv-python': 'cv2',
    'scikit-learn': 'sklearn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'numpy': 'numpy',
    'pillow': 'PIL',
    'captum': 'captum',
}
for dist_name, mod_name in required_packages.items():
    try:
        __import__(mod_name)
    except ImportError:
        pip_install(dist_name)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
import numpy as np
import random
from PIL import Image
import cv2
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
from captum.attr import IntegratedGradients
from typing import Dict

In [2]:
class Config:
    TRAIN_PATH = "cotton_split/Train"
    TEST_PATH = "cotton_split/Test"
    
    # Dataset info
    CLASS_NAMES = ['bacterial_blight', 'curl_virus', 'fusarium_wilt', 'healthy']
    NUM_CLASSES = 4
    
    # Training parameters
    SEED = 42
    BATCH_SIZE = 32
    NUM_EPOCHS_FUSION = 3
    EARLY_STOPPING_PATIENCE = 8
    LEARNING_RATE_FUSION = 1e-3
    WEIGHT_DECAY = 1e-4
    
    # Model architecture
    EMBED_DIM = 512
    DROPOUT = 0.3
    
    # XAI parameters
    XAI_NUM_SAMPLES = 30
    XAI_N_STEPS = 10  
    # Output
    OUTPUT_DIR = "cotton_disease_xai_results"
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 
                         ('mps' if torch.backends.mps.is_available() else 'cpu'))

    # ── Memory management ─────────────────────────────────────────────────
    # XAI_N_STEPS / XAI_NUM_SAMPLES defined above; keep n_steps low for VRAM
    GRAD_ACCUM_STEPS: int = 4      # gradient accumulation steps
    AMP_ENABLED: bool = True       # automatic mixed precision (fp16 forward)
    OFFLOAD_BACKBONES: bool = False  # backbones stay on CPU; no edge branch VRAM issue

In [3]:
# Set random seeds
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True   # auto-tune CUDA kernels

set_seed(Config.SEED)

# Create output directories
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
for sub in ('models', 'confusion_matrices', 'visualizations', 'reports', 'xai_analysis'):
    os.makedirs(os.path.join(Config.OUTPUT_DIR, sub), exist_ok=True)

print(f"Device: {Config.DEVICE}")
print(f"Output directory: {Config.OUTPUT_DIR}")
print(f"Classes: {Config.CLASS_NAMES}")

#  TRANSFORM DEFINITIONS 

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    normalize,
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

#  SINGLE-BRANCH DATASET 

class SingleBranchDataset(Dataset):
    def __init__(self, dataset, transform):
        self.dataset   = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return self.transform(img), label

#  BACKBONE MODELS 

def create_backbone_efficientnet_b4(pretrained=True):
    weights = models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b4(weights=weights)
    feature_dim = model.classifier[1].in_features
    model.classifier = nn.Identity()
    return model, feature_dim

def create_backbone_convnext_tiny(pretrained=True):
    weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
    model = models.convnext_tiny(weights=weights)
    feature_dim = model.classifier[2].in_features
    model.classifier = nn.Identity()
    return model, feature_dim

#  FUSION STRATEGIES 

class ConcatFusion(nn.Module):
    def __init__(self, d1, d2, num_classes, embed_dim=512, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(d1 + d2, embed_dim),
            nn.BatchNorm1d(embed_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.BatchNorm1d(embed_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes)
        )
    def forward(self, f1, f2):
        return self.classifier(torch.cat([f1, f2], dim=1))


class AdaptiveFusion(nn.Module):
    def __init__(self, d1, d2, num_classes, embed_dim=512, dropout=0.3):
        super().__init__()
        self.proj1 = nn.Sequential(nn.Linear(d1, embed_dim), nn.BatchNorm1d(embed_dim), nn.ReLU())
        self.proj2 = nn.Sequential(nn.Linear(d2, embed_dim), nn.BatchNorm1d(embed_dim), nn.ReLU())
        self.alpha = nn.Parameter(torch.ones(1) * 0.5)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.BatchNorm1d(embed_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes)
        )
    def forward(self, f1, f2):
        alpha = torch.sigmoid(self.alpha)
        return self.classifier(alpha * self.proj1(f1) + (1 - alpha) * self.proj2(f2))


class GatedFusion(nn.Module):
    def __init__(self, d1, d2, num_classes, embed_dim=512, dropout=0.3):
        super().__init__()
        self.proj1 = nn.Linear(d1, embed_dim)
        self.proj2 = nn.Linear(d2, embed_dim)
        self.gate  = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.BatchNorm1d(embed_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes)
        )
    def forward(self, f1, f2):
        p1, p2 = self.proj1(f1), self.proj2(f2)
        gate = self.gate(torch.cat([p1, p2], dim=1))
        return self.classifier(gate * p1 + (1 - gate) * p2)


class AttentionFusion(nn.Module):
    def __init__(self, d1, d2, num_classes, embed_dim=512, dropout=0.3):
        super().__init__()
        self.proj1     = nn.Linear(d1, embed_dim)
        self.proj2     = nn.Linear(d2, embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads=4,
                                               dropout=dropout, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.BatchNorm1d(embed_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes)
        )
    def forward(self, f1, f2):
        combined = torch.stack([self.proj1(f1), self.proj2(f2)], dim=1)  # (B, 2, E)
        attended, _ = self.attention(combined, combined, combined)
        return self.classifier(attended.mean(dim=1))


class WeightedSumFusion(nn.Module):
    def __init__(self, d1, d2, num_classes, embed_dim=512, dropout=0.3):
        super().__init__()
        self.w1 = nn.Parameter(torch.ones(1))
        self.w2 = nn.Parameter(torch.ones(1))
        self.classifier = nn.Sequential(
            nn.Linear(d1 + d2, embed_dim),
            nn.BatchNorm1d(embed_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.BatchNorm1d(embed_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes)
        )
    def forward(self, f1, f2):
        return self.classifier(torch.cat([torch.sigmoid(self.w1)*f1,
                                          torch.sigmoid(self.w2)*f2], dim=1))


FUSION_STRATEGIES = {
    'concat':       ConcatFusion,
    'adaptive':     AdaptiveFusion,
    'gated':        GatedFusion,
    'attention':    AttentionFusion,
    'weighted_sum': WeightedSumFusion,
}

#  DUAL-BACKBONE FUSION MODEL 

class DualBackboneModel(nn.Module):
    def __init__(self, backbone1, backbone2, fusion_class, d1, d2, num_classes,
                 backbone_device='cpu', fusion_device=None, **fusion_kwargs):
        super().__init__()
        self.backbone_device = torch.device(backbone_device)
        self.fusion_device   = torch.device(fusion_device or Config.DEVICE)
        self.xai_mode        = False

        self.backbone1 = backbone1.to(self.backbone_device)
        self.backbone2 = backbone2.to(self.backbone_device)
        self.fusion    = fusion_class(d1, d2, num_classes, **fusion_kwargs)

        for p in self.backbone1.parameters(): p.requires_grad = False
        for p in self.backbone2.parameters(): p.requires_grad = False
        self.backbone1.eval()
        self.backbone2.eval()

    def train(self, mode=True):
        super().train(mode)
        self.backbone1.eval()
        self.backbone2.eval()
        return self

    def enable_xai_mode(self):
        """Move backbones to GPU for Captum gradient tracing."""
        self.xai_mode = True
        self.backbone1 = self.backbone1.to(self.fusion_device)
        self.backbone2 = self.backbone2.to(self.fusion_device)
        clear_gpu()

    def disable_xai_mode(self):
        """Return backbones to CPU after XAI pass."""
        self.xai_mode = False
        self.backbone1 = self.backbone1.to(self.backbone_device)
        self.backbone2 = self.backbone2.to(self.backbone_device)
        clear_gpu()

    def forward(self, x):
        """x: (B, 3, H, W) — same image fed to both backbones."""
        if self.xai_mode:
            f1 = self.backbone1(x)
            f2 = self.backbone2(x)
        else:
            with torch.no_grad():
                f1 = self.backbone1(x.to(self.backbone_device)).to(self.fusion_device)
                f2 = self.backbone2(x.to(self.backbone_device)).to(self.fusion_device)

        if f1.dim() == 4: f1 = f1.mean(dim=[2, 3])
        if f2.dim() == 4: f2 = f2.mean(dim=[2, 3])
        return self.fusion(f1, f2)

#  XAI EXPLAINER 

class ModelExplainer:

    def __init__(self, model, device):
        self.model  = model
        self.device = device
        self.ig     = IntegratedGradients(model)

    def _ensure_batch(self, t):
        return t.unsqueeze(0).to(self.device) if t.dim() == 3 else t.to(self.device)

    def explain(self, img, target_class=None):
        self.model.eval()
        x = self._ensure_batch(img)
        if target_class is None:
            with torch.no_grad():
                target_class = int(self.model(x).argmax(dim=1).item())

        attributions, delta = self.ig.attribute(
            inputs=x,
            target=target_class,
            return_convergence_delta=True,
            n_steps=Config.XAI_N_STEPS,
            internal_batch_size=1,
        )
        with torch.no_grad():
            conf = F.softmax(self.model(x), dim=1).max().item()

        return {
            'attributions':      attributions,
            'model_confidence':  conf,
            'target_class':      target_class,
            'convergence_delta': float(delta.detach().abs().mean().item()),
        }

    def importance_map(self, img, target_class=None):
        result = self.explain(img, target_class)
        attr = result['attributions'].abs().mean(dim=1).detach().cpu().numpy()[0]
        attr = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)
        return attr, result

#  XAI METRICS EVALUATOR 

class XAIMetricsEvaluator:

    def __init__(self, models_dict: Dict[str, nn.Module], device):
        self.models    = models_dict
        self.device    = device
        self.explainers = {n: ModelExplainer(m, device) for n, m in models_dict.items()}
        self.results   = {}

    def compute_faithfulness(self, model, img, target_class):
        model.eval()
        x = img.unsqueeze(0).to(self.device) if img.dim() == 3 else img.to(self.device)
        ig = IntegratedGradients(model)
        with torch.no_grad():
            probs = F.softmax(model(x), dim=1)
        attrs = ig.attribute(inputs=x, target=int(target_class),
                             n_steps=Config.XAI_N_STEPS, internal_batch_size=1)
        return {
            'attr_mean': torch.abs(attrs).mean().item(),
            'attr_std':  torch.abs(attrs).std().item(),
            'confidence': probs.max().item(),
        }

    def evaluate_all_strategies(self, test_loader, num_samples=20):
        print("\n" + "=" * 70)
        print("XAI EVALUATION ACROSS FUSION STRATEGIES")
        print("=" * 70)

        samples = []
        for imgs, labels in test_loader:
            for i in range(len(labels)):
                samples.append((imgs[i], labels[i]))
                if len(samples) >= num_samples:
                    break
            if len(samples) >= num_samples:
                break

        for name, explainer in self.explainers.items():
            print(f"\n  Evaluating {name.upper()}...")
            confidences, attr_means = [], []
            for img, lbl in samples:
                res = explainer.explain(img, lbl.item())
                confidences.append(res['model_confidence'])
                faith = self.compute_faithfulness(explainer.model, img, lbl.item())
                attr_means.append(faith['attr_mean'])
                clear_gpu()

            self.results[name] = {
                'avg_confidence':   float(np.mean(confidences)),
                'avg_attr_strength': float(np.mean(attr_means)),
                'overall_xai_score': float(np.mean(confidences)),
            }
        return self.results

    def generate_comparison_table(self):
        print("\n" + "=" * 70)
        print("XAI METRICS COMPARISON TABLE")
        print("=" * 70)
        print(f"\n{'Strategy':<15} {'Avg Confidence':<18} {'Attr Strength':<15} {'XAI Score':<10}")
        print("-" * 60)
        for name, m in sorted(self.results.items(),
                               key=lambda x: x[1]['overall_xai_score'], reverse=True):
            print(f"{name:<15} {m['avg_confidence']:>14.3f}   "
                  f"{m['avg_attr_strength']:>13.5f}   {m['overall_xai_score']:>8.3f}")
        print("-" * 60)
        return self.results

#  VISUALIZATION 

def visualize_explanation(explainer, img, save_path, class_names, true_label, pred_label):

    attr_map, result = explainer.importance_map(img)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    img_np = img.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * np.array([0.229, 0.224, 0.225])
              + np.array([0.485, 0.456, 0.406])).clip(0, 1)

    axes[0].imshow(img_np)
    axes[0].set_title('Input Image', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(attr_map, cmap='hot')
    axes[1].set_title('Attribution Map', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046)

    axes[2].imshow(img_np)
    axes[2].imshow(attr_map, cmap='Reds', alpha=0.6)
    axes[2].set_title('Overlay', fontsize=12, fontweight='bold')
    axes[2].axis('off')

    true_cls = class_names[true_label] if 0 <= true_label < len(class_names) else '?'
    pred_cls = class_names[pred_label] if 0 <= pred_label < len(class_names) else '?'
    conf     = result['model_confidence'] * 100
    plt.suptitle(f'True: {true_cls} | Predicted: {pred_cls} | Confidence: {conf:.1f}%',
                 fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    return cm


def plot_training_curves(history, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train', linewidth=2)
    ax1.plot(history['val_loss'],   label='Val',   linewidth=2)
    ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(history['train_acc'], label='Train', linewidth=2)
    ax2.plot(history['val_acc'],   label='Val',   linewidth=2)
    ax2.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


def create_xai_summary_report(xai_results, output_dir, class_names):
    path = os.path.join(output_dir, 'xai_analysis_report.md')
    best = max(xai_results.items(), key=lambda x: x[1]['overall_xai_score'])[0]
    with open(path, 'w') as f:
        f.write("# XAI Analysis — Cotton Disease Classification\n\n")
        f.write(f"## Best Strategy: **{best.upper()}**\n\n")
        f.write("| Strategy | XAI Score | Avg Confidence | Attr Strength |\n")
        f.write("|----------|-----------|----------------|---------------|\n")
        for name, m in sorted(xai_results.items(),
                               key=lambda x: x[1]['overall_xai_score'], reverse=True):
            f.write(f"| {name} | {m['overall_xai_score']:.3f} | "
                    f"{m['avg_confidence']:.3f} | {m['avg_attr_strength']:.5f} |\n")
    print(f"\nXAI report saved to: {path}")
    return path

#  TRAINING FUNCTIONS 

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    accum = Config.GRAD_ACCUM_STEPS
    optimizer.zero_grad()

    for idx, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=_use_amp):
            outputs = model(imgs)
            loss    = criterion(outputs, labels) / accum

        _scaler.scale(loss).backward()

        if (idx + 1) % accum == 0 or (idx + 1) == len(loader):
            _scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            _scaler.step(optimizer)
            _scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accum
        _, pred = outputs.max(1)
        total   += labels.size(0)
        correct += pred.eq(labels).sum().item()

        if idx % 10 == 0:
            alloc = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
            print(f"    Batch {idx}/{len(loader)}  "
                  f"Loss: {loss.item()*accum:.4f}  GPU: {alloc:.2f} GB")

    return total_loss / max(len(loader), 1), 100.0 * correct / max(total, 1)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            probs   = F.softmax(outputs, dim=1)
            total_loss += loss.item()
            _, pred = outputs.max(1)
            total   += labels.size(0)
            correct += pred.eq(labels).sum().item()
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return (total_loss / max(len(loader), 1),
            100.0 * correct / max(total, 1),
            all_preds, all_labels, all_probs)

#  MEMORY UTILITIES 

import gc

def setup_cuda_memory():
    if not torch.cuda.is_available():
        return
    import torch as _t
    major, minor = [int(x) for x in _t.__version__.split(".")[:2]
                    if x.isdigit() or x.split("a")[0].isdigit()][:2]
    opts = "max_split_size_mb:128"
    if (major, minor) >= (2, 1):
        opts += ",expandable_segments:True"
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", opts)
    torch.cuda.empty_cache()
    gc.collect()
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    alloc = torch.cuda.memory_allocated(0) / 1e9
    print(f"[MEM] GPU: {total:.1f} GB total | {alloc:.2f} GB allocated")

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

setup_cuda_memory()
_use_amp = Config.AMP_ENABLED and torch.cuda.is_available()
_scaler  = torch.cuda.amp.GradScaler(enabled=_use_amp)

#  UNCERTAINTY ESTIMATOR 

class ConformalUncertaintyEstimator:
    def __init__(self, model, device, alpha=0.1):
        self.model  = model
        self.device = device
        self.alpha  = alpha
        self.q_hat  = 1.0

    @torch.no_grad()
    def calibrate(self, cal_loader, max_batches=20):
        self.model.eval()
        scores = []
        for i, (imgs, labels) in enumerate(cal_loader):
            if i >= max_batches: break
            imgs, labels = imgs.to(self.device), labels.to(self.device)
            probs = torch.softmax(self.model(imgs), dim=1)
            scores.extend((1.0 - probs[range(len(labels)), labels]).cpu().tolist())
        n = len(scores)
        level = min(1.0, np.ceil((n + 1) * (1 - self.alpha)) / n)
        self.q_hat = float(np.quantile(scores, level))
        clear_gpu()
        return self.q_hat

    def uncertainty(self, img):
        self.model.eval()
        x = img.unsqueeze(0).to(self.device) if img.dim() == 3 else img.to(self.device)
        with torch.no_grad():
            score = 1.0 - torch.softmax(self.model(x), dim=1).max().item()
        return score

    def is_high_uncertainty(self, img):
        return self.uncertainty(img) > self.q_hat

#  CDAC METRIC 

class CrossDiseaseAttributionConsistency:
    def __init__(self, model, device, num_classes, class_names, n_steps=10):
        self.model       = model
        self.device      = device
        self.num_classes = num_classes
        self.class_names = class_names
        self.n_steps     = n_steps
        self.ig          = IntegratedGradients(model)

    def _attr_vector(self, img, target):
        self.model.eval()
        x = img.unsqueeze(0).to(self.device) if img.dim() == 3 else img.to(self.device)
        attrs = self.ig.attribute(inputs=x, target=target,
                                  n_steps=self.n_steps, internal_batch_size=1)
        vec  = attrs.abs().flatten().cpu().numpy()
        norm = np.linalg.norm(vec) + 1e-8
        clear_gpu()
        return vec / norm

    def compute(self, loader, max_per_class=3):
        class_attrs = {i: [] for i in range(self.num_classes)}
        counts      = {i: 0  for i in range(self.num_classes)}

        for imgs, labels in loader:
            if all(counts[i] >= max_per_class for i in range(self.num_classes)):
                break
            for j in range(len(labels)):
                lbl = int(labels[j].item())
                if counts[lbl] >= max_per_class: continue
                class_attrs[lbl].append(self._attr_vector(imgs[j], lbl))
                counts[lbl] += 1

        intra, centroids = {}, {}
        for cls_id, vecs in class_attrs.items():
            name = self.class_names[cls_id] if cls_id < len(self.class_names) else str(cls_id)
            if len(vecs) < 2:
                intra[name] = 1.0
                centroids[cls_id] = vecs[0] if vecs else np.zeros(1)
                continue
            mat      = np.stack(vecs)
            centroid = mat.mean(axis=0)
            centroid /= (np.linalg.norm(centroid) + 1e-8)
            centroids[cls_id] = centroid
            norms    = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-8
            cos_mat  = (mat @ mat.T) / (norms * norms.T)
            mask     = ~np.eye(len(vecs), dtype=bool)
            intra[name] = float(cos_mat[mask].mean())

        ids = list(centroids.keys())
        inter_dists = [float(centroids[ids[i]] @ centroids[ids[j]])
                       for i in range(len(ids)) for j in range(i+1, len(ids))]
        inter_sep  = 1.0 - float(np.mean(inter_dists)) if inter_dists else 0.0
        mean_intra = float(np.mean(list(intra.values())))

        return {
            'intra_class_consistency':  intra,
            'mean_intra_consistency':   mean_intra,
            'inter_class_separability': inter_sep,
            'cdac_score':               0.5 * mean_intra + 0.5 * inter_sep,
            'samples_per_class':        {self.class_names[k] if k < len(self.class_names)
                                         else str(k): v for k, v in counts.items()},
        }

    @staticmethod
    def print_report(r, name):
        print(f"\n{'─'*55}")
        print(f"  CDAC — {name.upper()}")
        print(f"{'─'*55}")
        print(f"  CDAC Score            : {r['cdac_score']:.4f}")
        print(f"  Mean Intra-Consistency: {r['mean_intra_consistency']:.4f}")
        print(f"  Inter-Separability    : {r['inter_class_separability']:.4f}")
        for cls, val in r['intra_class_consistency'].items():
            print(f"    {cls:<25} {val:.4f}")
        print(f"{'─'*55}")


Device: cuda
Output directory: cotton_disease_xai_results
Classes: ['bacterial_blight', 'curl_virus', 'fusarium_wilt', 'healthy']
[MEM] GPU: 12.5 GB total | 0.00 GB allocated


In [4]:
#  MAIN TRAINING PIPELINE 

def main():
    print("\n" + "=" * 70)
    print("COTTON DISEASE CLASSIFICATION WITH XAI")
    print("=" * 70)

    # 1. Load datasets
    print("\n1. Loading datasets...")
    try:
        train_dataset = ImageFolder(Config.TRAIN_PATH, transform=None)
        test_dataset  = ImageFolder(Config.TEST_PATH,  transform=None)
        print(f"   Training samples : {len(train_dataset)}")
        print(f"   Testing  samples : {len(test_dataset)}")
        print(f"   Classes found    : {train_dataset.classes}")
        if train_dataset.classes != Config.CLASS_NAMES:
            Config.CLASS_NAMES  = train_dataset.classes
            Config.NUM_CLASSES  = len(Config.CLASS_NAMES)
    except Exception as e:
        print(f"   Error: {e}"); return None, None, None

    # 2. Split & wrap datasets
    train_size  = int(0.8 * len(train_dataset))
    val_size    = len(train_dataset) - train_size
    train_sub, val_sub = random_split(train_dataset, [train_size, val_size])
    print(f"   Train subset: {train_size}  |  Val subset: {val_size}")

    train_ds = SingleBranchDataset(train_sub, train_transform)
    val_ds   = SingleBranchDataset(val_sub,   val_transform)
    test_ds  = SingleBranchDataset(test_dataset, val_transform)

    # 3. DataLoaders  (num_workers=4: safe — no cv2 in transforms)
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE,
                              shuffle=True,  num_workers=4,
                              pin_memory=torch.cuda.is_available(),
                              persistent_workers=True)
    val_loader   = DataLoader(val_ds, batch_size=Config.BATCH_SIZE * 2,
                              shuffle=False, num_workers=4,
                              pin_memory=torch.cuda.is_available(),
                              persistent_workers=True)
    test_loader  = DataLoader(test_ds, batch_size=Config.BATCH_SIZE * 2,
                              shuffle=False, num_workers=4,
                              pin_memory=torch.cuda.is_available(),
                              persistent_workers=True)
    print(f"   Train batches: {len(train_loader)}  "
          f"Val batches: {len(val_loader)}  "
          f"Test batches: {len(test_loader)}")

    # 4. Backbones (frozen, CPU-resident)
    print("\n2. Creating backbone models...")
    try:
        backbone1, d1 = create_backbone_efficientnet_b4(pretrained=True)
        backbone2, d2 = create_backbone_convnext_tiny(pretrained=True)
    except Exception as e:
        print(f"   Backbone load failed: {e}"); return None, None, None
    for p in backbone1.parameters(): p.requires_grad = False
    for p in backbone2.parameters(): p.requires_grad = False
    backbone1.eval(); backbone2.eval()
    print(f"   EfficientNet-B4 dim: {d1}  |  ConvNeXt-Tiny dim: {d2}")

    # 5. Train each fusion strategy
    print("\n3. Training fusion strategies...")
    all_results, trained_models = {}, {}
    criterion = nn.CrossEntropyLoss()

    for strat_name, fusion_cls in FUSION_STRATEGIES.items():
        print(f"\n{'='*60}")
        print(f"Training: {strat_name.upper()} Fusion")
        print(f"{'='*60}")
        try:
            model = DualBackboneModel(
                backbone1=backbone1, backbone2=backbone2,
                fusion_class=fusion_cls, d1=d1, d2=d2,
                num_classes=Config.NUM_CLASSES,
                backbone_device='cpu', fusion_device=Config.DEVICE,
                embed_dim=Config.EMBED_DIM, dropout=Config.DROPOUT,
            )
            model.fusion = model.fusion.to(Config.DEVICE)
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"   Trainable parameters: {trainable:,}")

            optimizer = optim.AdamW(model.fusion.parameters(),
                                    lr=Config.LEARNING_RATE_FUSION,
                                    weight_decay=Config.WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=5)

            best_val_acc, best_state, patience_ctr = 0, None, 0
            history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
            t0 = time.time()

            for epoch in range(Config.NUM_EPOCHS_FUSION):
                te = time.time()
                tr_loss, tr_acc = train_epoch(model, train_loader, optimizer,
                                              criterion, Config.DEVICE)
                va_loss, va_acc, _, _, _ = evaluate(model, val_loader,
                                                    criterion, Config.DEVICE)
                scheduler.step(va_acc)
                for k, v in zip(('train_loss','train_acc','val_loss','val_acc'),
                                 (tr_loss, tr_acc, va_loss, va_acc)):
                    history[k].append(v)

                if va_acc > best_val_acc:
                    best_val_acc = va_acc
                    best_state = {k: v.detach().cpu().clone()
                                  for k, v in model.state_dict().items()}
                    patience_ctr = 0
                    print(f"  Epoch {epoch+1}: New best val acc: {va_acc:.2f}%")
                else:
                    patience_ctr += 1

                print(f"  Epoch [{epoch+1}/{Config.NUM_EPOCHS_FUSION}] "
                      f"Train {tr_loss:.4f}/{tr_acc:.2f}%  "
                      f"Val {va_loss:.4f}/{va_acc:.2f}%  "
                      f"({time.time()-te:.1f}s)")

                if patience_ctr >= Config.EARLY_STOPPING_PATIENCE:
                    print(f"  Early stopping at epoch {epoch+1}"); break

            if best_state: model.load_state_dict(best_state)
            print(f"\n  Training done in {time.time()-t0:.1f}s")

            # Test evaluation
            _, test_acc, test_preds, test_labels, _ = evaluate(
                model, test_loader, criterion, Config.DEVICE)
            f1 = f1_score(test_labels, test_preds, average='weighted')

            per_class = {}
            for i, name in enumerate(Config.CLASS_NAMES):
                mask = np.array(test_labels) == i
                if mask.sum() > 0:
                    per_class[name] = (np.array(test_preds)[mask] == i).sum() / mask.sum() * 100

            all_results[strat_name] = {
                'best_val_acc': best_val_acc, 'test_acc': test_acc,
                'test_f1': f1, 'history': history,
                'predictions': test_preds, 'labels': test_labels,
                'per_class_acc': per_class,
            }
            trained_models[strat_name] = model
            print(f"  {strat_name.upper()} — Test Acc: {test_acc:.2f}%  F1: {f1:.4f}")

            # Save artefacts
            plot_confusion_matrix(test_labels, test_preds, Config.CLASS_NAMES,
                f'{strat_name.upper()} Fusion',
                os.path.join(Config.OUTPUT_DIR, 'confusion_matrices', f'{strat_name}_cm.png'))
            plot_training_curves(history,
                os.path.join(Config.OUTPUT_DIR, 'visualizations', f'{strat_name}_curves.png'))
            torch.save({'strategy': strat_name, 'model_state': best_state,
                        'test_accuracy': test_acc, 'test_f1': f1},
                os.path.join(Config.OUTPUT_DIR, 'models', f'{strat_name}_best.pth'))

        except Exception as e:
            import traceback; traceback.print_exc()
            print(f"  Error with {strat_name}: {e}")
        finally:
            clear_gpu()

    if not all_results:
        print("\n No strategies completed."); return None, None, None

    # 6. Summary table
    print("\n" + "=" * 70)
    print("CLASSIFICATION RESULTS SUMMARY")
    print("=" * 70)
    sorted_res = sorted(all_results.items(), key=lambda x: x[1]['test_acc'], reverse=True)
    print(f"\n{'Strategy':<20} {'Test Acc':<12} {'F1':<10} {'Best Val':<10}")
    print("-" * 55)
    for name, r in sorted_res:
        print(f"{name:<20} {r['test_acc']:.2f}%      {r['test_f1']:.4f}    {r['best_val_acc']:.2f}%")
    best_strat = sorted_res[0][0]
    print(f"\n BEST: {best_strat.upper()} — {sorted_res[0][1]['test_acc']:.2f}% test accuracy")

    # 7. CDAC
    print("\n" + "=" * 70)
    print("CDAC METRIC")
    print("=" * 70)
    cdac_all = {}
    if trained_models:
        for strat, model in trained_models.items():
            print(f"\n  [{strat.upper()}] Computing CDAC...")
            model.enable_xai_mode()
            cdac = CrossDiseaseAttributionConsistency(
                model, Config.DEVICE, Config.NUM_CLASSES, Config.CLASS_NAMES,
                n_steps=Config.XAI_N_STEPS)
            res = cdac.compute(test_loader, max_per_class=3)
            cdac_all[strat] = res
            CrossDiseaseAttributionConsistency.print_report(res, strat)
            model.disable_xai_mode()
            clear_gpu()

    # 8. XAI evaluation
    print("\n" + "=" * 70)
    print("XAI EVALUATION")
    print("=" * 70)
    xai_results = None
    if trained_models:
        # Enable XAI mode for all models
        for m in trained_models.values(): m.enable_xai_mode()

        evaluator   = XAIMetricsEvaluator(trained_models, Config.DEVICE)
        xai_results = evaluator.evaluate_all_strategies(
            test_loader, num_samples=Config.XAI_NUM_SAMPLES)
        evaluator.generate_comparison_table()

        # Visualise best strategy
        sample_imgs, sample_labels = next(iter(test_loader))
        sample_img  = sample_imgs[0]
        sample_lbl  = int(sample_labels[0].item())

        for strat, model in trained_models.items():
            with torch.no_grad():
                pred = int(model(sample_img.unsqueeze(0).to(Config.DEVICE)).argmax(1).item())
            explainer = ModelExplainer(model, Config.DEVICE)
            visualize_explanation(
                explainer, sample_img,
                os.path.join(Config.OUTPUT_DIR, 'xai_analysis', f'{strat}_attribution.png'),
                Config.CLASS_NAMES, sample_lbl, pred)
            clear_gpu()

        for m in trained_models.values(): m.disable_xai_mode()
        create_xai_summary_report(xai_results, Config.OUTPUT_DIR, Config.CLASS_NAMES)

    # 9. Save results JSON
    with open(os.path.join(Config.OUTPUT_DIR, 'reports', 'results.json'), 'w') as f:
        json.dump({n: {'test_accuracy': float(r['test_acc']),
                       'test_f1': float(r['test_f1']),
                       'best_val_accuracy': float(r['best_val_acc']),
                       'per_class_accuracy': {k: float(v) for k,v in r['per_class_acc'].items()}}
                   for n, r in all_results.items()}, f, indent=4)

    # Best model classification report
    best_r = all_results[best_strat]
    print(f"\nClassification Report — {best_strat.upper()}:")
    print(classification_report(best_r['labels'], best_r['predictions'],
                                 target_names=Config.CLASS_NAMES))

    print("\n" + "=" * 70)
    print("EXPERIMENT COMPLETE")
    print("=" * 70)
    print(f"Results saved to: {Config.OUTPUT_DIR}")
    return all_results, best_strat, xai_results


if __name__ == "__main__":
    results, best_model, xai_results = main()



COTTON DISEASE CLASSIFICATION WITH XAI

1. Loading datasets...
   Training samples : 1366
   Testing  samples : 344
   Classes found    : ['bacterial_blight', 'curl_virus', 'fussarium_wilt', 'healthy']
   Train subset: 1092  |  Val subset: 274
   Train batches: 35  Val batches: 5  Test batches: 6

2. Creating backbone models...
   EfficientNet-B4 dim: 1792  |  ConvNeXt-Tiny dim: 768

3. Training fusion strategies...

Training: CONCAT Fusion
   Trainable parameters: 1,445,124
    Batch 0/35  Loss: 1.4605  GPU: 0.05 GB
    Batch 10/35  Loss: 0.5394  GPU: 0.06 GB
    Batch 20/35  Loss: 0.3549  GPU: 0.06 GB
    Batch 30/35  Loss: 0.2582  GPU: 0.06 GB
  Epoch 1: New best val acc: 92.70%
  Epoch [1/3] Train 0.5765/80.22%  Val 0.3424/92.70%  (73.5s)
    Batch 0/35  Loss: 0.1587  GPU: 0.06 GB
    Batch 10/35  Loss: 0.2405  GPU: 0.06 GB
    Batch 20/35  Loss: 0.1807  GPU: 0.06 GB
    Batch 30/35  Loss: 0.1399  GPU: 0.06 GB
  Epoch 2: New best val acc: 98.91%
  Epoch [2/3] Train 0.1878/95.88%  